In [ ]:
def main(datasources, start_date, end_date):
    import dai

    sql = """
    WITH base AS (
        SELECT
            f.date,
            f.instrument,
            e.industry_level1_code,
            f.net_active_buy_amount_main,
            f.turn
        FROM bigalpha_2026_factorlib f
        JOIN bigalpha_2026_exposure e USING (date, instrument)
        WHERE f.net_active_buy_amount_main IS NOT NULL
          AND f.turn IS NOT NULL
          AND e.industry_level1_code IS NOT NULL
    ),
    ranked AS (
        SELECT
            date,
            instrument,
            PERCENT_RANK() OVER (
                PARTITION BY date, industry_level1_code
                ORDER BY net_active_buy_amount_main
            ) AS active_buy_rank,
            PERCENT_RANK() OVER (
                PARTITION BY date, industry_level1_code
                ORDER BY turn
            ) AS turnover_rank
        FROM base
    )
    SELECT
        date,
        instrument,
        active_buy_rank * (1.0 - turnover_rank) - 0.25 AS factor
    FROM ranked
    ORDER BY date, instrument
    """

    df = dai.query(
        sql,
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()

    return df[["date", "instrument", "factor"]]
